<a href="https://colab.research.google.com/github/cuiandrew08-lab/LiDARFusionLearning/blob/main/FusionTrain.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install --force-reinstall numpy==1.26.4

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.0/61.0 kB 2.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.0/18.0 MB 57.3 MB/s eta 0:00:00
  Attempting uninstall: numpy
    Found existing installation: numpy 2.0.2
    Uninstalling numpy-2.0.2:
      Successfully uninstalled numpy-2.0.2
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
jax 0.7.2 requires numpy>=2.0, but you have numpy 1.26.4 which is incompatible.
jaxlib 0.7.2 requires numpy>=2.0, but you have numpy 1.26.4 which is incompatible.
opencv-contrib-python 4.13.0.92 requires numpy>=2; python_version >= "3.9", but you have numpy 1.26.4 which is incompatible.
opencv-python 4.13.0.92 requires numpy>=2; python_version >= "3.9", but you have numpy 1.26.4 which is incompatible.
opencv-python-headless 4.13.0.92 requires numpy>=2; python_version >= "3.9", but you have numpy 1.26.4 which is i

In [1]:
import os
import numpy as np

from google.colab import drive
drive.mount("/content/drive", force_remount = False)

import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
import torchvision.transforms as transforms
from torchvision.transforms import v2
from tqdm.notebook import tqdm

import random

from torch.utils.data import Dataset, DataLoader

#import tensorflow as tf

TORCH_version = torch.__version__.split('+')[0]
CUDA_version = '128'

#!pip install torch-scatter torch-sparse torch-cluster -f https://data.pyg.org/whl/torch-{TORCH_version}+cu{CUDA_version}.html

!pip install torch-sparse torch-scatter -f https://data.pyg.org/whl/torch-{TORCH_version}+cu{CUDA_version}.html

!pip install torch-geometric

#import torch_sparse
#import torch_scatter

import torch_geometric

from PIL import Image

from scipy.ndimage import maximum_filter
from scipy.spatial._qhull import ConvexHull

import sys

import matplotlib.pyplot as plt

Mounted at /content/drive
Looking in links: https://data.pyg.org/whl/torch-2.10.0+cu128.html
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.4/5.4 MB 38.4 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.9/10.9 MB 52.7 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 64.4/64.4 kB 2.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 19.5 MB/s eta 0:00:00


/usr/local/lib/python3.12/dist-packages/torch_geometric/__init__.py:4: UserWarning: An issue occurred while importing 'torch-scatter'. Disabling its usage. Stacktrace: Could not load this library: /usr/local/lib/python3.12/dist-packages/torch_scatter/_scatter_cuda.so
  import torch_geometric.typing
/usr/local/lib/python3.12/dist-packages/torch_geometric/__init__.py:4: UserWarning: An issue occurred while importing 'torch-sparse'. Disabling its usage. Stacktrace: Could not load this library: /usr/local/lib/python3.12/dist-packages/torch_sparse/_convert_cuda.so
  import torch_geometric.typing


In [2]:
!npx degit google-research-datasets/Objectron/objectron objectron

!pip install --force-reinstall opencv-python-headless==4.9.0.80 &> /dev/null
!pip install nuscenes-devkit &> /dev/null

⠙⠹⠸⠼⠴⠦⠧⠇⠏Need to install the following packages:
degit@3.8.0
Ok to proceed? (y) y

⠙⠹⠸⠼⠴> cloned google-research-datasets/Objectron#HEAD to objectron
⠙npm notice
npm notice New major version of npm available! 10.8.2 -> 12.0.2
npm notice Changelog: https://github.com/npm/cli/releases/tag/v12.0.2
npm notice To update run: npm install -g npm@12.0.2
npm notice
⠙

In [3]:
sys.path.insert(0, '/content')

from objectron.dataset.iou import IoU as IoU3d
from objectron.dataset.box import Box as BoxIoU

from nuscenes.nuscenes import NuScenes
from nuscenes.utils.data_classes import LidarPointCloud, Box
from nuscenes.eval.detection.utils import category_to_detection_name
from nuscenes.utils.geometry_utils import points_in_box

nusc_root = "/content/drive/MyDrive/LiDARFusion/nuscenes/datanuscenes"

nusc = NuScenes(version='v1.0-mini', dataroot=nusc_root, verbose=True)

Loading NuScenes tables for version v1.0-mini...
23 category,
8 attribute,
4 visibility,
911 instance,
12 sensor,
120 calibrated_sensor,
31206 ego_pose,
8 log,
10 scene,
404 sample,
31206 sample_data,
18538 sample_annotation,
4 map,
Done loading in 6.798 seconds.
Reverse indexing ...
Done reverse indexing in 0.1 seconds.


In [4]:
from pyquaternion import Quaternion

sys.path.insert(0, '/content/drive/MyDrive/LiDARFusion')

sys.path.append('/content/objectron/')

import lidartrainlibrary as ltb

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [9]:
def load_cameras(nusc, sample):
  cam_channels = ['CAM_FRONT','CAM_FRONT_RIGHT','CAM_BACK_RIGHT',
                'CAM_BACK','CAM_BACK_LEFT','CAM_FRONT_LEFT']

  images, intrinsics, cam_to_ego = [], [], []

  for ch in cam_channels:
    cam_token = sample['data'][ch]
    cam_data = nusc.get('sample_data', cam_token)
    img = Image.open(nusc.get_sample_data_path(cam_token))
    calib = nusc.get('calibrated_sensor', cam_data['calibrated_sensor_token'])
    K = np.array(calib['camera_intrinsic'])          # 3x3
    cam_to_ego_pose = (calib['translation'], calib['rotation'])  # quaternion
    images.append(img); intrinsics.append(K); cam_to_ego.append(cam_to_ego_pose)

  return images, intrinsics, cam_to_ego


In [33]:
test_scene = nusc.scene[1]
token_0 = test_scene["first_sample_token"]

img_sample = nusc.get("sample", token_0)

cam = load_cameras(nusc, img_sample)


In [24]:
def process_images(images):

  augmented_transform = v2.Compose([
      v2.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2),
      v2.ToImage(),
      v2.ToDtype(torch.float32, scale=True),
      v2.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
  ])

  img_list = []

  for image in images:
    image_out = augmented_transform(image)
    img_list.append(image_out)

  out = torch.stack(img_list, dim=0)

  return out #camera dims are 900 x 1600


In [34]:
processed = process_images(cam[0])